# Export Hang Analysis

Diagnose a **hung or very slow** Genesys Cloud **resource export** from **`TF_LOG=json`** trace output.

Looks for:

- **Stuck export** — `Started processing for resource` without matching `Collected resource`
- **SDK retry storms** — repeated `invocation_retry_after` (429 rate limits)
- **SDK 404 loops** — API object not found (retry retry retry)
- **Tail export activity** — what the exporter was doing at the end of the log

Capture while export is running or immediately after killing it:

```bash
export TF_LOG=json
export TF_LOG_PATH=export-hang.log
# optional: GENESYSCLOUD_SDK_DEBUG=true GENESYSCLOUD_SDK_DEBUG_FORMAT=Json
# run your export (terraform / genesyscloud exporter)
export TERRAFORM_LOG_PATH=export-hang.log
```

For completed exports use `export/analysis.ipynb`. Run `whatisit.ipynb` first if unsure.


In [ ]:
import sys
from pathlib import Path

_nb_root = Path.cwd()
if (_nb_root.parent / "notebook_setup.py").is_file():
    sys.path.insert(0, str(_nb_root.parent))
elif (_nb_root / "notebook_setup.py").is_file():
    sys.path.insert(0, str(_nb_root))

import notebook_setup

notebook_setup.setup()

import matplotlib.pyplot as plt
import pandas as pd

import commonlib.config as cfg
import commonlib.prep_export_data as prep_export_data
import commonlib.prep_hang_data as hang


In [ ]:
TAIL_MINUTES = 5
MIN_REPEAT = 3
WORKFLOW = "export"

c = cfg.Config()
print(c.TERRAFORM_LOG_PATH)

classification, records, counters = hang.load_hang_scan(c.TERRAFORM_LOG_PATH, tail_minutes=TAIL_MINUTES)
if not records:
    raise ValueError("No JSON log lines found. Capture with TF_LOG=json and run whatisit.ipynb first.")

normalized_records = prep_export_data.normalize_records(records)

if not classification.is_export and not counters.export_starts:
    raise ValueError(
        "No export activity found. Use export/hang-analysis.ipynb on an export TF_LOG capture, "
        "or run whatisit.ipynb to pick the right notebook."
    )

summary = hang.hang_summary_for_workflow(
    counters,
    tail_minutes=TAIL_MINUTES,
    workflow=WORKFLOW,
    min_count=MIN_REPEAT,
    classification=classification,
)


## Summary

In [ ]:
print(f"Parsed lines: {summary['parsed_lines']:,}")
if summary['duration_minutes'] is not None:
    print(f"Log span: {summary['duration_minutes']:.1f} minutes")
if summary['first_timestamp']:
    print(f"Time range: {summary['first_timestamp']} → {summary['last_timestamp']}")
print(f"Tail window: last {summary['tail_minutes']:.0f} minutes")
print()
print(f"Primary suspect: {summary['primary_summary']}")
if summary['primary_detail']:
    print(f"  {summary['primary_detail']}")
print()
print(
    f"Export: {summary['export_resources_started']:,} resources started, "
    f"{summary['export_resources_completed']:,} completed, "
    f"{summary['sdk_retry_endpoints']:,} retry endpoints, "
    f"{summary['sdk_404_endpoints']:,} 404 endpoints"
)
if summary['sdk_status_codes']:
    print(f"SDK status codes: {summary['sdk_status_codes']}")


## Ranked hang suspects

In [ ]:
pd.DataFrame([
    {"category": v.category, "summary": v.summary, "detail": v.detail, "score": v.score}
    for v in summary["verdicts"]
]) if summary["verdicts"] else "No strong hang patterns detected (try lowering MIN_REPEAT)."


## Stuck or repeating exports

Resources where export started but did not finish.

In [ ]:
hang.export_imbalance_dataframe(counters, min_gap=1).head(20)


## Tail export messages

In [ ]:
hang.tail_export_messages_dataframe(counters, top_n=15)


## SDK retry storms

Endpoints returning `invocation_retry_after` — rate limits or SDK backoff.

In [ ]:
df_retries = hang.sdk_retry_dataframe(counters, min_count=MIN_REPEAT)
df_retries.head(20) if not df_retries.empty else "No SDK retry storms found."


In [ ]:
if not df_retries.empty:
    plot_df = df_retries.head(15).sort_values("retry_responses")
    plt.figure(figsize=(12, 6))
    plt.barh(plot_df["method_url"], plot_df["retry_responses"], color="tab:red")
    plt.xlabel("responses with retry_after")
    plt.title("SDK retry storms")
    plt.tight_layout()


## SDK 404 not found

Missing Genesys Cloud objects — often causes endless retry.

In [ ]:
df_404 = hang.sdk_not_found_dataframe(counters, min_count=2)
df_404.head(20) if not df_404.empty else "No repeated SDK 404 responses found."


## Tail activity (last few minutes)

What the log was doing right before capture.

In [ ]:
hang.tail_messages_dataframe(counters, top_n=25)
